In [3]:
import os
os.environ["NUMBA_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMBA_DISABLE_JIT"] = "0"

In [4]:
import numpy as np
from numba import njit, prange, float32, uint8
from src.constants import *
from time import time, perf_counter


In [5]:
N = len(NEURON_NAMES)
ALPHA_L = 250
td = np.arange(1, ALPHA_L + 1, dtype=np.float32)
alpha = (td / 30) * np.exp((30 - td) / 30)   

cue_wave = np.zeros(TMAX, dtype=np.float32)
go_wave = np.zeros_like(cue_wave)
cue_wave[EPOCHS['sample'][0]:EPOCHS['sample'][1]] = CUE_STRENGTH
go_wave[EPOCHS['response'][0]:EPOCHS['response'][0] + GO_DURATION] = GO_STRENGTH


# --------------------------------------------------------------------
# Hand‑crafted weights -------------------------------------------------
# --------------------------------------------------------------------
new_jh_weights = [
    ("Somat", "ALMprep", 40),
    ("Somat", "MSN1", 220),
    ("MSN1", "SNR1", -90),
    ("SNR1", "VMprep", -10),
    ("VMprep", "ALMprep", 70),
    ("ALMprep", "VMprep", 80),
    ("ALMprep", "MSN2", 320),
    ("MSN2", "SNR2", -50),
    ("SNR2", "VMresp", -100),
    ("PPN", "THALgo", 60),
    ("THALgo", "ALMinter", 55),
    ("ALMinter", "ALMprep", -50),
    ("THALgo", "ALMresp", 30),
    ("ALMresp", "MSN3", 320),
    ("MSN3", "SNR3", -90),
    ("SNR3", "VMresp", -50),
    ("VMresp", "ALMresp", 85),
    ("ALMresp", "VMresp", 90),
]

# --------------------------------------------------------------------
# Build weight matrix -------------------------------------------------
# --------------------------------------------------------------------
N = len(NEURON_NAMES)
W = np.zeros((N, N), dtype=np.float32)
for pre, post, w in new_jh_weights:
    i = NEURON_NAMES.index(pre)
    j = NEURON_NAMES.index(post)
    W[i, j] += w

pass_ids = [NEURON_NAMES.index(x) for x in ["VMresp", "ALMresp", "SNR3"]]
pass_ids = np.array(pass_ids)
print(pass_ids)

[13 12 11]


In [6]:
# CREATING CRITERION
conditions = []
for condition in CRITERIA:
    condition_criteria = []
    for neuron_name, neuron in CRITERIA[condition].items():
        idx = NEURON_NAMES.index(neuron_name)
        baseline = np.ones(TMAX, np.uint8) if neuron_name in TONICALLY_ACTIVE_NEURONS else np.zeros(TMAX, np.uint8)
        start = neuron["interval"][0]
        end = neuron["interval"][1]
        target_status = neuron["io"]
        # print(idx, neuron_name, baseline)
        for i in baseline:
            if target_status == "off":
                baseline[start:end] = 0
            elif target_status == "on":
                baseline[start:end] = 1

        baseline = baseline.reshape(TMAX//BIN_SIZE, BIN_SIZE)
        baseline = np.sum(baseline, axis=1,dtype=np.uint32)
        baseline = (baseline != 0).astype(np.uint8)

        condition_criteria.append((neuron_name, idx, baseline))
    condition_criteria = sorted(condition_criteria, key=lambda tup: tup[1])
    conditions.append(condition_criteria)

crit_Exp, crit_Cont = conditions

crit_indices = np.array([neu[1] for neu in crit_Cont])
crit_Exp = np.vstack([neu[2] for neu in crit_Exp])
crit_Cont = np.vstack([neu[2] for neu in crit_Cont])


In [7]:
# Takes the state at t and updates world to t+1. Returns spikes from step

@njit(parallel=False, fastmath=True, cache=True)
def step_kernel(V, U, Ibuf, t_ptr,
                a, b, vreset, d, k, vr, vt, vpeak, C, E, 
                W, alpha):
    n, L = V.size, alpha.size
    spk  = np.zeros(n, dtype=np.uint8)

    # integrate -------------------------------------------------------
    for i in range(n):
        I = Ibuf[t_ptr, i]
        dV  = (k[i]*(V[i]-vr[i])*(V[i]-vt[i]) - U[i] + I + E[i]) / C[i]
        dU  = a[i]*(b[i]*(V[i]-vr[i]) - U[i])
        V[i] += dV
        U[i] += dU
        if V[i] >= vpeak[i]:
            V[i]  = vreset[i]
            U[i] += d[i]
            spk[i] = 1          # Double check the formula to make sure it aint wonky

    # distribute PSC --------------------------------------------------
    if np.sum(spk) > 0:
        post_I = spk.astype(np.float32) @ W                   # dense GEMV
        t_next = (t_ptr + 1) % L
        for k_shift in range(L):
            Ibuf[(t_next + k_shift) % L, :] += post_I * alpha[k_shift]

    Ibuf[t_ptr,:] = 0.0
    return spk, (t_ptr + 1) % L

In [8]:
@njit(fastmath=True, cache=True)
def score_bin(curr_bin_results, crit_matrix, crit_indices, bin_idx, pass_ids):
    score = 0
    for i in range(len(crit_indices)):
        idx = crit_indices[i]
        if curr_bin_results[idx] == crit_matrix[i, bin_idx]:
            score += 1
        elif (bin_idx * BIN_SIZE > 3500) and (idx in pass_ids):
            score += 1
    return score


In [28]:
# ────────────────────────────────────────────────────────────────────
# 2.  Simulation + scoring
# ────────────────────────────────────────────────────────────────────

@njit(fastmath=True, cache=True)
def simulate(W, 
            a, b, vreset, d, k, vr, vt, vpeak, C, E, 
            alpha, cue_wave, go_wave, 
            crit_Exp, crit_Cont, crit_indices, pass_ids,
            tmax, 
            control, 
            return_full 
            ):
    
    W = np.ascontiguousarray(W)
    V = np.full(N, -60.0, np.float32)
    U = np.zeros_like(V, np.float32)
    Ibuf = np.zeros((ALPHA_L, N), dtype=np.float32)
    HIST = np.zeros((N, BIN_SIZE), np.uint8) # 99?
    if return_full:
        temp_full_hist = np.zeros((N, tmax), np.uint8) # 99?

    score = 0
    t_ptr   = 0
    bin = 0

    for t in range(tmax):

        if control == False:
            Ibuf[t_ptr,0] += cue_wave[t]
        Ibuf[t_ptr,7] += go_wave[t]
    
        spk, t_ptr = step_kernel(V, U, Ibuf, t_ptr,
                                 a, b, vreset, d, k, vr, vt, vpeak, C, E, 
                                 W, alpha)

        if return_full:
            temp_full_hist[:,t] = spk 

        cidx = t % BIN_SIZE
        HIST[:,cidx] = spk
        # bit-pack history
        if cidx == (BIN_SIZE - 1):
            curr_bin_results = (np.sum(HIST, axis=1) >= 1).astype(np.uint8)
            crits = crit_Exp if (control == False) else crit_Cont
            score += score_bin(curr_bin_results,crits, crit_indices, bin, pass_ids)
            bin += 1

    return score, (temp_full_hist if return_full else None)


In [10]:
start = perf_counter()
simulate(W, a, b, vreset, d, k, vr, vt, vpeak, C, E,
         alpha, cue_wave, go_wave,
         crit_Exp, crit_Cont, crit_indices, pass_ids,
         5000, False, False)
mid = perf_counter()
# print(mid-start)
# run_batch(W, a, b, vreset, d, k, vr, vt, vpeak, C, E,
#           alpha, cue_wave, go_wave,
#           crit_Exp, crit_Cont, crit_indices, pass_ids,
#           TMAX)
end = perf_counter()
print(f'Total time: {end - start:.3f}s')


Total time: 0.553s


In [11]:
@njit(cache=True)
def run_batch(W, a, b, vreset, d, k, vr, vt, vpeak, C, E,
              alpha, cue_wave, go_wave,
              crit_Exp, crit_Cont, crit_indices, pass_ids,
              tmax, NTRIALS=10):

    total_time = 0.0
    for i in range(NTRIALS):
        s1, _ = simulate(W, a, b, vreset, d, k, vr, vt, vpeak, C, E,
                         alpha, cue_wave, go_wave,
                         crit_Exp, crit_Cont, crit_indices, pass_ids,
                         tmax, False, False)
        s2, _ = simulate(W, a, b, vreset, d, k, vr, vt, vpeak, C, E,
                         alpha, cue_wave, go_wave,
                         crit_Exp, crit_Cont, crit_indices, pass_ids,
                         tmax, True, False)
    return s1, s2

In [32]:
start = perf_counter()
s1,s2=run_batch(W, a, b, vreset, d, k, vr, vt, vpeak, C, E,
          alpha, cue_wave, go_wave,
          crit_Exp, crit_Cont, crit_indices, pass_ids,
          TMAX)
end = perf_counter()
print(f'Total time: {end - start:.3f}s')


Total time: 0.316s


# Runtimes for 10 runs
- No JIT: 6 seconds
- Step Kernel @njit(parallel=False, fastmath=True, cache=True) without prange: 0.592s
- Step Kernel @njit(parallel=False, fastmath=True, cache=True) with prange: 0.555s
- Step Kernel @njit(parallel=True, fastmath=True, cache=True) without prange: 22.615s 
- Step Kernel @njit(parallel=True, fastmath=True, cache=True) with prange: 21.592s
- Step Kernel and Score Bin JIT: 0.569s
- Step Kernel and Score Bin and Simulate JIT: HUNG, but then .388s

In [13]:
print(s1, s2)


470 498


# Config param setup
```
GA_CONFIG={
    "small": {
        "NUM_GENERATIONS" : 5,
        "POP_SIZE" : 50,
        "MUT_RATE" : 0.3,
        "MUT_SIGMA" : 0.5,
        "RANK_DEPTH" : 25,
        "ELITE_SIZE" : 5,
        "CROSSOVER_POINT" : None, # Randomly selecting all genes
        "DNA_BOUNDS" : [0,500]
    },...
}
```

In [ ]:
# Generating random matrices

def create_population(size, upper_bound, random=True):
    base_vector = np.ones(len(ACTIVE_SYNAPSES), dtype=np.int32)
    for idx, conn in enumerate(ACTIVE_SYNAPSES):
        if conn[0] in INHIBITORY_NEURONS:
            base_vector[idx] *= -1
    
    vectors = []
    if random:
        while len(vectors) < size:
            random_vector = base_vector*np.random.randint(0,upper_bound,len(base_vector))
            # if random_vector in previous_dnas:      #implement distance function here
            vectors.append(random_vector)      
    vectors = np.array(vectors)
    
    return vectors


def create_matrices(dna_vectors: np.array):
    Ws = np.zeros((len(dna_vectors), N, N), dtype=np.float32)
    for idx, vector in enumerate(dna_vectors):
        for conn, w in zip(ACTIVE_SYNAPSES,vector):
            row = NEURON_NAMES.index(conn[0])
            col = NEURON_NAMES.index(conn[1])
            Ws[idx, row, col] += w
    return Ws

    

cfg="small"
pop = create_population(10, GA_CONFIG[cfg]["DNA_BOUNDS"][1], random=True)

print(pop[0:3])
print("\n\n")

matrices= create_matrices(pop)
print(matrices[0:3])

tmax=5000


[[ 243  335   65  168 -333 -194  -10 -248 -290  -28 -240  -79 -312 -367
  -104 -274  -13 -106 -396  380  476  346  197  153  332  103   51  237
   -89 -190 -472  429  203  253  309  391  189  278  -77 -292 -305 -370
  -351 -384  463  265 -354 -477  242   39   13  241  327]
 [ 455  192  290  489 -140 -250 -476    0 -145 -490 -267 -407 -367 -377
  -446 -339 -426 -485 -493  481    9  286  470  369  189  147   39   17
  -185 -106 -280  370  217  474  497  310  228  235 -480  -76 -453 -167
  -330  -56  231  114 -229 -189  440  356  181  365  344]
 [  70  221  132  122 -271 -202 -277  -75 -378  -51 -151 -120 -276 -430
  -162 -429 -306  -23 -315  473  415  235  194  207  285  495  187  286
   -44 -381 -195   45  356  360  269  208  231  169  -21 -350 -268 -461
  -229 -417  265  390 -435 -343  353  481   65   96  405]]



[[[   0.  335.    0.    0.  243.   65.    0.    0.    0.    0.  168.
      0.    0.    0.]
  [   0.    0. -333.    0.    0.  -77. -194.    0.    0.    0. -292.
    -10.    0.

In [34]:

#Need to make this a 2D array instead of a an array of arrays
@njit(parallel=False, fastmath=True, cache=True)
def evaluate_population(population_matrices,
                        a, b, vreset, d, k, vr, vt, vpeak, C, E, 
                        alpha, cue_wave, go_wave, 
                        crit_Exp, crit_Cont, crit_indices, pass_ids,
                        tmax, 
                        return_full):
    
    vectors_scores = np.zeros((len(population_matrices), 2), dtype=np.int32)
    for idx, W in enumerate(population_matrices):
        exp_score, _ = simulate(W, a, b, vreset, d, k, vr, vt, vpeak, C, E,
                            alpha, cue_wave, go_wave,
                            crit_Exp, crit_Cont, crit_indices, pass_ids,
                            tmax, False, return_full)
        cont_score, _ = simulate(W, a, b, vreset, d, k, vr, vt, vpeak, C, E,
                            alpha, cue_wave, go_wave,
                            crit_Exp, crit_Cont, crit_indices, pass_ids,
                            tmax, True, return_full)
        vectors_scores[idx,0] += exp_score
        vectors_scores[idx,1] += cont_score    

    return vectors_scores 

In [27]:

scores = evaluate_population(matrices, 
                    a, b, vreset, d, k, vr, vt, vpeak, C, E,
                    alpha, cue_wave, go_wave,
                    crit_Exp, crit_Cont, crit_indices, pass_ids,
                    tmax, False)
print(scores)

[[316 374]
 [336 411]
 [308 330]
 [357 498]
 [409 490]
 [349 452]
 [313 301]
 [283 253]
 [307 386]
 [331 411]]


In [47]:
start = perf_counter()
s1,s2=run_batch(W, a, b, vreset, d, k, vr, vt, vpeak, C, E,
          alpha, cue_wave, go_wave,
          crit_Exp, crit_Cont, crit_indices, pass_ids,
          TMAX)
end = perf_counter()
print(f'Total time: {end - start:.3f}s')


KeyboardInterrupt: 

In [ ]:

# ────────────────────────────────────────────────────────────────────
# 3.  Example run
# ────────────────────────────────────────────────────────────────────
if __name__ == "__main__":

    t0 = time.time()
    simulate(W, cue_wave, go_wave)
    # score, hist = simulate(W, cue_wave, go_wave)
    # print(f"TOTAL score: {score}")
    print(f"Wall-time: {1000*(time.time()-t0):.1f} ms")

    # unpack if needed
    # bits = np.unpackbits(hist, axis=1)[:, :TMAX]   # shape (N,TMAX)
    # print("Neuron-0 first 40 ms spikes:", bits[0, :40].tolist())